# GroundingDINO + Tracker video demo

This notebook loads a test video, runs GroundingDINO detections frame by frame, feeds them into the tracker, writes a tracked output video, and then renders a browser-friendly MP4.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

import base64
import shutil
import subprocess

from IPython.display import HTML, display
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

from tracker.utils import load_gdino_model, run_tracker_on_video

Repo root: /home/jdill/projects/open-vocabulary-object-tracking
Using device: cuda


/home/jdill/miniconda3/envs/ovtrack/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
CONFIG_PATH = "../external/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
WEIGHTS_PATH = "../external/GroundingDINO/weights/groundingdino_swint_ogc.pth"

VIDEO_PATH = "demo2.mp4"
OUTPUT_VIDEO_PATH = "output.mp4"

LABELS = ["shirt", "pants", "head"] # ["person"]
BOX_THRESHOLD = 0.2
MAX_FRAMES = None

In [3]:
model = load_gdino_model(CONFIG_PATH, WEIGHTS_PATH, device=DEVICE)

/home/jdill/miniconda3/envs/ovtrack/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased


/home/jdill/projects/open-vocabulary-object-tracking/tracker/utils.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


In [ ]:
tracked_path = run_tracker_on_video(
    model=model,
    video_path=VIDEO_PATH,
    output_path=OUTPUT_VIDEO_PATH,
    labels=LABELS,
    device=DEVICE,
    box_threshold=BOX_THRESHOLD,
    max_frames=MAX_FRAMES,
    draw_trails=True,
    show_progress=True,
)
print("Saved", tracked_path)

Tracking video:   0%|          | 0/493 [00:00<?, ?frame/s]

/home/jdill/miniconda3/envs/ovtrack/lib/python3.10/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/jdill/miniconda3/envs/ovtrack/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/jdill/miniconda3/envs/ovtrack/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/jdill/projects/open-vocabulary-object-tracking/external/GroundingDINO/groundingdino/models/

In [ ]:
def reencode_mp4_for_notebook(input_path: str, output_path: str | None = None) -> str:
    input_path = str(input_path)
    if output_path is None:
        p = Path(input_path)
        output_path = str(p.with_name(p.stem + "_browser.mp4"))

    if shutil.which("ffmpeg") is None:
        print("ffmpeg not found; using original file")
        return input_path

    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        output_path,
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return output_path


def display_mp4(path: str, width: int = 900):
    video_bytes = Path(path).read_bytes()
    encoded = base64.b64encode(video_bytes).decode("ascii")
    html = f'''
    <video width="{width}" controls>
        <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    '''
    display(HTML(html))

In [ ]:
browser_video_path = reencode_mp4_for_notebook(OUTPUT_VIDEO_PATH)
print("Displaying:", browser_video_path)
display_mp4(browser_video_path)